## Import

In [4]:
import psycopg2
from psycopg2.extras import execute_values
import random
import datetime
from dateutil.relativedelta import relativedelta
from collections import defaultdict

## Lấy ra danh sách invoices

In [5]:
conn = psycopg2.connect(
    host="aws-1-ap-south-1.pooler.supabase.com",
    database="postgres",
    user="postgres.rruavcjmtgpxyznzwkhw",
    password="khaibaolocnguyen",
    port=5432
)
cur = conn.cursor()

In [6]:
conn.rollback()
cur.execute("SELECT invoice_id, customer_id, total_price FROM invoice")
invoices = cur.fetchall()

# invoices là danh sách tuple: [(invoice_id, customer_id, price), ...]
for inv in invoices:
    print(inv)

('8a724108-1ce7-4138-8ec5-bc07ed0ac3cf', 'f0019289-a95b-4f54-b663-bea43f7d9c1d', Decimal('23210117'))
('bfe5ae5e-683c-43fc-b013-150bd1bd876b', '1a58413a-7512-4830-95f5-43484d89eec5', Decimal('6943011'))
('e985ab9b-dadd-4b6c-bb7e-00c8d7f3e306', '00fb3bbe-bc8e-43d7-a744-01674757ddc5', Decimal('17722710'))
('a44e6f8a-cb83-42fc-af3f-8c4cc0591c46', 'a4e1f1a7-cb7e-4e95-bfec-07ba568cee11', Decimal('15928630'))
('b12182bc-b479-4dc2-a92b-64024d93bfa8', '3da2776b-8b3e-4356-96bd-fc07bf3b86ca', Decimal('8438304'))
('af5c0e58-4c78-4023-a6a7-b5b6962645ae', '7ab375f9-0955-4858-b494-98fbd4386c24', Decimal('9562195'))
('808dcb65-92bd-47df-b286-94f31bc96844', '9a07b4dd-78d2-4323-afcd-6297e5c7d0c4', Decimal('15288177'))
('9acaf87f-5fde-4d5f-b0fd-8b6eca79d05a', '31545f4c-7519-4135-b453-b0ac487c3a77', Decimal('1803096'))
('37cb5328-4a0e-4259-a8c0-06de03a9b3c0', '623bc093-1dfd-4b6c-b162-aee4785f2c07', Decimal('8313708'))
('b84d1a91-fd25-4320-8e2f-dc0c7e6a3348', '7b105d91-bf92-4367-8125-3544536cd0ac', Decima

## Tính tổng tiền tiêu của từng Customer

In [7]:
total_by_customer = defaultdict(int)

for invoice_id, customer_id, price in invoices:
    total_by_customer[customer_id] += int(price)

# Kết quả
for customer_id, total in total_by_customer.items():
    print(f"Customer {customer_id} total: {total}")
print(len(total_by_customer))

Customer f0019289-a95b-4f54-b663-bea43f7d9c1d total: 33150720
Customer 1a58413a-7512-4830-95f5-43484d89eec5 total: 18450276
Customer 00fb3bbe-bc8e-43d7-a744-01674757ddc5 total: 33674566
Customer a4e1f1a7-cb7e-4e95-bfec-07ba568cee11 total: 25243535
Customer 3da2776b-8b3e-4356-96bd-fc07bf3b86ca total: 41646355
Customer 7ab375f9-0955-4858-b494-98fbd4386c24 total: 15870229
Customer 9a07b4dd-78d2-4323-afcd-6297e5c7d0c4 total: 24039087
Customer 31545f4c-7519-4135-b453-b0ac487c3a77 total: 19322209
Customer 623bc093-1dfd-4b6c-b162-aee4785f2c07 total: 18101473
Customer 7b105d91-bf92-4367-8125-3544536cd0ac total: 21448474
Customer f6573871-086d-49f8-bd58-ac880745083d total: 18987421
Customer e36d7f84-726f-44a5-8273-daab6f357b95 total: 27759480
Customer 51273354-224e-4986-a4fd-2b6d6c264039 total: 37001828
Customer e330ea3b-be3b-4c48-ab8d-d5e641e27149 total: 26927930
Customer c62c0bd9-5234-4578-8629-b017938c61ad total: 42565949
Customer 1288f85f-cfcf-44bf-8f5f-c318069abe46 total: 19973033
Customer

In [8]:
conn.rollback()
cur.execute("SELECT level_id, target_threshold FROM membershiplevel")
membershiplevels = cur.fetchall()

def format_membership_level(level_id, target_threshold):
    return {
        "level_id": level_id,
        "target_threshold": target_threshold
    }

membershiplevels = [format_membership_level(level_id, target_threshold) for level_id, target_threshold in membershiplevels]
print(membershiplevels)

[{'level_id': '7c5b193e-ff68-42b2-bfb5-0a9eb7dcfa8f', 'target_threshold': Decimal('3000')}, {'level_id': 'a332a04f-df54-4f50-b68e-291abe8cf5bd', 'target_threshold': Decimal('7000')}, {'level_id': 'e5d3eff2-b534-4a8d-ab19-0223c730b435', 'target_threshold': Decimal('15000')}, {'level_id': '2fff97c5-50ce-4af8-9d43-6f02529f9c99', 'target_threshold': Decimal('30000')}]


In [9]:
tmp = []
for customer_id, total in total_by_customer.items():
    eligible_level = None
    for level in membershiplevels:
        if int(total / 1000) >= int(level["target_threshold"]):
            eligible_level = level
    if eligible_level:
        print(f"Customer {customer_id}: {eligible_level['level_id']}")
        query = """
          UPDATE customeraccount
          SET level_id = %s
          WHERE customer_id = %s
        """
        cur.execute(query, (eligible_level['level_id'], customer_id))
    conn.commit()

Customer f0019289-a95b-4f54-b663-bea43f7d9c1d: 2fff97c5-50ce-4af8-9d43-6f02529f9c99
Customer 1a58413a-7512-4830-95f5-43484d89eec5: e5d3eff2-b534-4a8d-ab19-0223c730b435
Customer 00fb3bbe-bc8e-43d7-a744-01674757ddc5: 2fff97c5-50ce-4af8-9d43-6f02529f9c99
Customer a4e1f1a7-cb7e-4e95-bfec-07ba568cee11: e5d3eff2-b534-4a8d-ab19-0223c730b435
Customer 3da2776b-8b3e-4356-96bd-fc07bf3b86ca: 2fff97c5-50ce-4af8-9d43-6f02529f9c99
Customer 7ab375f9-0955-4858-b494-98fbd4386c24: e5d3eff2-b534-4a8d-ab19-0223c730b435
Customer 9a07b4dd-78d2-4323-afcd-6297e5c7d0c4: e5d3eff2-b534-4a8d-ab19-0223c730b435
Customer 31545f4c-7519-4135-b453-b0ac487c3a77: e5d3eff2-b534-4a8d-ab19-0223c730b435
Customer 623bc093-1dfd-4b6c-b162-aee4785f2c07: e5d3eff2-b534-4a8d-ab19-0223c730b435
Customer 7b105d91-bf92-4367-8125-3544536cd0ac: e5d3eff2-b534-4a8d-ab19-0223c730b435
Customer f6573871-086d-49f8-bd58-ac880745083d: e5d3eff2-b534-4a8d-ab19-0223c730b435
Customer e36d7f84-726f-44a5-8273-daab6f357b95: e5d3eff2-b534-4a8d-ab19-0223c

In [10]:
for customer_id, total in total_by_customer.items():
    print(f"Customer {customer_id} total: {total}")
    query = """
      UPDATE customeraccount
      SET loyalty_score = %s
      WHERE customer_id = %s
"""
    cur.execute(query, (int(total / 1000), customer_id))
    conn.commit()

Customer f0019289-a95b-4f54-b663-bea43f7d9c1d total: 33150720
Customer 1a58413a-7512-4830-95f5-43484d89eec5 total: 18450276
Customer 00fb3bbe-bc8e-43d7-a744-01674757ddc5 total: 33674566
Customer a4e1f1a7-cb7e-4e95-bfec-07ba568cee11 total: 25243535
Customer 3da2776b-8b3e-4356-96bd-fc07bf3b86ca total: 41646355
Customer 7ab375f9-0955-4858-b494-98fbd4386c24 total: 15870229
Customer 9a07b4dd-78d2-4323-afcd-6297e5c7d0c4 total: 24039087
Customer 31545f4c-7519-4135-b453-b0ac487c3a77 total: 19322209
Customer 623bc093-1dfd-4b6c-b162-aee4785f2c07 total: 18101473
Customer 7b105d91-bf92-4367-8125-3544536cd0ac total: 21448474
Customer f6573871-086d-49f8-bd58-ac880745083d total: 18987421
Customer e36d7f84-726f-44a5-8273-daab6f357b95 total: 27759480
Customer 51273354-224e-4986-a4fd-2b6d6c264039 total: 37001828
Customer e330ea3b-be3b-4c48-ab8d-d5e641e27149 total: 26927930
Customer c62c0bd9-5234-4578-8629-b017938c61ad total: 42565949
Customer 1288f85f-cfcf-44bf-8f5f-c318069abe46 total: 19973033
Customer